In [1]:
# !pip install -q opencv-python

In [3]:
import os, io, base64, subprocess, time, json
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

import cv2
from PIL import Image
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini")
vision = ChatOpenAI(model="gpt-4o-mini", max_tokens=200)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

VIDEO_PATH = Path("data/video/earnings_briefing.mp4")
WORK_DIR = Path("day53_video")
WORK_DIR.mkdir(exist_ok=True)

In [4]:
path = str(VIDEO_PATH)
cap = cv2.VideoCapture(path)

In [5]:
fps = cap.get(cv2.CAP_PROP_FPS)
print(fps)

24.0


In [7]:
n_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
print(n_frames)

717.0


In [13]:
n_frames // fps

29.0

In [10]:
w = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
print(w)

720.0


In [11]:
h = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
print(h)

1018.0


In [12]:
def video_info(path):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return {'fps' : round(fps,2), 'n_frames' : n_frames, 'duration_sec' : round(n_frames / fps if fps else 0, 2), 'width':w, 'height': h}

In [18]:
def sample_frames(video_path, every_sec, out_dir = None):
    out_dir = Path(out_dir or WORK_DIR / 'frames')
    out_dir.mkdir(parents=True, exist_ok=True)
    info_ = video_info(video_path)
    cap = cv2.VideoCapture(video_path)
    items = []
    sec = 0.0
    i = 0
    
    while sec <= info_['duration_sec']:
        cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
        ok, frame = cap.read()
        if not ok:
            break
        
        p = out_dir / f"frame_{i:03d}_{int(sec*1000):07d}ms.jpg"
#         cv2.imwrite(str(p), frame)
        items.append({'path' : str(p), 'second' : round(sec,2)})      
        
        sec += every_sec
        i +=1
    cap.release()
    return items

In [61]:
frames = sample_frames(str(VIDEO_PATH), every_sec = 10)

In [ ]:
# ffmpeg

In [19]:
!ffmpeg

ffmpeg version 6.1.1 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 11.2.0 (Anaconda gcc)
  configuration: --prefix=/home/task_176243685929256/conda-bld/ffmpeg_1762437043313/_h_env_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_ --cc=/home/task_176243685929256/conda-bld/ffmpeg_1762437043313/_build_env/bin/x86_64-conda-linux-gnu-cc --ar=/home/task_176243685929256/conda-bld/ffmpeg_1762437043313/_build_env/bin/x86_64-conda-linux-gnu-ar --nm=/home/task_176243685929256/conda-bld/ffmpeg_1762437043313/_build_env/bin/x86_64-conda-linux-gnu-nm --ranlib=/home/task_176243685929256/conda-bld/ffmpeg_1762437043313/_build_env/bin/x86_64-conda-linux-gnu-ranlib --strip=/home/task_176243685929256/conda-bld/ffmpeg_1762437043313/_build_env/bin/x86_64-conda-linux-gnu-strip --disable-doc --enable-swresample --enable-swscale --enable-openssl --enable-l

In [22]:
def extract_audio(video_path, out_path):
    if Path(out_path).exists():
        return out_path
    
    cmd = ['ffmpeg', '-y', '-i', video_path, '-vn', '-c:a', 'libmp3lame', '-b:a', '128k', out_path]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-300:])
    return out_path

In [23]:
# ffmpeg -y -i 'briefing.mp4' -vn ...

In [55]:
audio_path = extract_audio(str(VIDEO_PATH), str(WORK_DIR / 'audio.mp3'))

In [56]:
def transcribe_with_segments(audio_path) -> dict:
    with open(audio_path, 'rb') as f:
        res = oai_audio.audio.transcriptions.create(
                model = 'whisper-1',
                file = f,
                response_format = 'verbose_json',
                language= 'ko'
        )
        
        segs = [{'id' : s.id, 'start' : s.start, 'end' : s.end, 'text' : s.text}
                 for s in (res.segments or [])]
    
    
    return {'text' : res.text, 'segments' : segs}

In [57]:
audio_seg = transcribe_with_segments(audio_path)

In [70]:
audio_seg['segments']

[{'id': 0,
  'start': 0.0,
  'end': 3.0799999237060547,
  'text': ' 오늘 분기 실적 회의를 시작하겠습니다.'},
 {'id': 1,
  'start': 3.0799999237060547,
  'end': 6.679999828338623,
  'text': ' 2025년 4분기 매출은 188억 원으로'},
 {'id': 2,
  'start': 6.679999828338623,
  'end': 9.920000076293945,
  'text': ' 전 분기 대비 16% 증가했습니다.'},
 {'id': 3,
  'start': 9.920000076293945,
  'end': 14.15999984741211,
  'text': ' 특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며'},
 {'id': 4,
  'start': 14.15999984741211,
  'end': 17.079999923706055,
  'text': ' 가장 큰 성장을 보였습니다.'},
 {'id': 5,
  'start': 17.079999923706055,
  'end': 19.040000915527344,
  'text': ' 영업이익은 24억 원으로'},
 {'id': 6,
  'start': 19.040000915527344,
  'end': 22.920000076293945,
  'text': ' 전년 동기 대비 35% 늘었습니다.'},
 {'id': 7,
  'start': 22.920000076293945,
  'end': 26.600000381469727,
  'text': ' 다음 분기는 동남아시아 시장 진출 본격화할 계획입니다.'}]

In [59]:
for s in audio_seg['segments'][:5]:
    print(f" [{s['start']} - {s['end']}] {s['text'][:50]}")

 [0.0 - 3.0799999237060547]  오늘 분기 실적 회의를 시작하겠습니다.
 [3.0799999237060547 - 6.679999828338623]  2025년 4분기 매출은 188억 원으로
 [6.679999828338623 - 9.920000076293945]  전 분기 대비 16% 증가했습니다.
 [9.920000076293945 - 14.15999984741211]  특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며
 [14.15999984741211 - 17.079999923706055]  가장 큰 성장을 보였습니다.


In [62]:
frames

[{'path': 'day53_video/frames/frame_000_0000000ms.jpg', 'second': 0.0},
 {'path': 'day53_video/frames/frame_001_0010000ms.jpg', 'second': 10.0},
 {'path': 'day53_video/frames/frame_002_0020000ms.jpg', 'second': 20.0}]

In [65]:
def _b64(path):
    return base64.b64encode(Path(path).read_bytes()).decode()

def caption_frames(frames:list, prompt:str= "이 프레임에 무엇이 보이는지 한 문장으로 써주세요") -> list:
    out = []
    for fr in frames:
        b = _b64(fr['path'])
        msg = HumanMessage(content = [
            {'type' : 'text', 'text' : prompt},
            {'type' : 'image_url', 'image_url' : {'url' : f"data:image/jpeg:base64,{b}"}}
        ])
        cap = vision.invoke([msg]).content
        out.append({**fr, "caption" : cap})
    return out

In [66]:
captioned = caption_frames(frames)

In [67]:
captioned

[{'path': 'day53_video/frames/frame_000_0000000ms.jpg',
  'second': 0.0,
  'caption': '이 문서는 Modu Tech의 2025년 4분기 실적 보고서로, 188억 원의 매출과 24억 원의 영업이익을 기록하며, 지난해 동기 대비 각각 35%와 42%의 성장을 이룬 내용을 담고 있습니다.'},
 {'path': 'day53_video/frames/frame_001_0010000ms.jpg',
  'second': 10.0,
  'caption': '이 문서는 Modu Tech의 2025년 4분기 실적 보고서로, 매출 188억 원과 영업이익 24억 원을 기록하며 전년동기 대비 35% 성장한 내용을 담고 있습니다.'},
 {'path': 'day53_video/frames/frame_002_0020000ms.jpg',
  'second': 20.0,
  'caption': '이 프레임에는 2024년과 2025년의 월별 매출 추이를 시장 평균과 비교한 그래프와 주요 KPI 정보가 포함되어 있습니다.'}]

In [68]:
# video rag
def build_video_documents(frames, segments, source)-> list:
    docs = []
    for fr in frames:
        docs.append(Document(
            page_content = fr['caption'],
            metadata = {'source' : source, 'modality' : 'frame', 'second' : fr['second'], 'frame_path' : fr['path']}
        ))
    for seg in segments:
        # {'text' : res.text, 'segments' : segs}
        docs.append(Document(
            page_content = seg['text'].strip(),
            metadata = {'source' : source, 'modality' : 'audio', 'start' : round(seg['start'],2), 'end' : round(seg['end'],2)}
        ))
    return docs

In [71]:
video_docs = build_video_documents(captioned, audio_seg['segments'], source=VIDEO_PATH.name)

In [72]:
len(video_docs)

11

In [73]:
for d in video_docs:
    if d.metadata['modality'] =='frame':
        print(f" [frame @ {d.metadata['second']}s] {d.page_content[:50]}")
    else:
        print(f" [audio {d.metadata['start']} - {d.metadata['end']}] {d.page_content[:50]}")

 [frame @ 0.0s] 이 문서는 Modu Tech의 2025년 4분기 실적 보고서로, 188억 원의 매출과 24
 [frame @ 10.0s] 이 문서는 Modu Tech의 2025년 4분기 실적 보고서로, 매출 188억 원과 영업이
 [frame @ 20.0s] 이 프레임에는 2024년과 2025년의 월별 매출 추이를 시장 평균과 비교한 그래프와 주요
 [audio 0.0 - 3.08] 오늘 분기 실적 회의를 시작하겠습니다.
 [audio 3.08 - 6.68] 2025년 4분기 매출은 188억 원으로
 [audio 6.68 - 9.92] 전 분기 대비 16% 증가했습니다.
 [audio 9.92 - 14.16] 특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며
 [audio 14.16 - 17.08] 가장 큰 성장을 보였습니다.
 [audio 17.08 - 19.04] 영업이익은 24억 원으로
 [audio 19.04 - 22.92] 전년 동기 대비 35% 늘었습니다.
 [audio 22.92 - 26.6] 다음 분기는 동남아시아 시장 진출 본격화할 계획입니다.


In [ ]:
# video_docs -> vectorstore indexing
# 매출 188억 -> 검색

In [74]:
vs = FAISS.from_documents(video_docs, embeddings)

In [75]:
vs.index.ntotal

11

In [76]:
hits = vs.similarity_search('매출 188억', k=3)

In [77]:
for h in hits:
    m = h.metadata
    tag = (f"[frame @ {m.get('second')}s]" if m['modality'] =='frame'
          else f"[audio {m.get('start')} - {m.get('end')}]")
    print(f" {tag} {h.page_content[:50]}")

 [audio 3.08 - 6.68] 2025년 4분기 매출은 188억 원으로
 [frame @ 10.0s] 이 문서는 Modu Tech의 2025년 4분기 실적 보고서로, 매출 188억 원과 영업이
 [audio 17.08 - 19.04] 영업이익은 24억 원으로


In [ ]:
# 검색 -> 이미지 -> 

In [78]:
def video_rag_visual(vs, query, k=4):
    cands = vs.similarity_search(query, k=k)
    text_blocks = []
    image_paths = []
    
    for d in cands:
        m = d.metadata
        if m['modality'] == 'frame':
            image_paths.append((m['second'], m['frame_path'], d.page_content))
        else:
            text_blocks.append(f"[audio {m['start']}-{m['end']}s] {d.page_content}")
    
    content = [{'type' : 'text',
               'text' : f"비디오에서 발췌한 프레임 이미지와 음성 transcribe을 참고하여 한국어로 답하세요.\n\n질문 : {query}"
               }]
    if text_blocks:
        content.append({'type' : 'text', 'text' : '[음성발췌]\n' + '\n'.join(text_blocks)})
    for sec, path, cap in image_paths:
        content.append({'type' : 'text', 'text' : f"\n[frame @ {sec}s] (caption : {cap})"})
        content.append({'type' : 'image_url', 'image_url' : {'url' : f"data:image/jpeg;base64,{_b64(path)}"}})
    
    msg = HumanMessage(content= content)
    return {'answer' : vision.invoke([msg]).content, 'n_frames' : len(image_paths), 'n_audio' : len(text_blocks)}

In [79]:
r = video_rag_visual(vs, '비디오 첫 장면에 어떤 차트가 보이나요?', k=3)

In [80]:
r

{'answer': '비디오의 첫 장면에서는 2024년과 2025년의 월별 매출 추이를 시장 평균과 비교한 그래프가 보입니다. 이 그래프는 2024년 4분기와 2025년 4분기의 매출 변화율, 주요 KPI 정보도 포함하고 있습니다.',
 'n_frames': 1,
 'n_audio': 2}

In [25]:
from langchain_community.document_loaders.parsers.audio import OpenAIWhisperParser
from langchain_core.document_loaders.blob_loaders import Blob


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [27]:
AUDIO_DIR = Path('data/audio')
audio_files = sorted(AUDIO_DIR.glob('*.mp3'))  # a1.mp3, a2.mp3 ..b1.mp3, ..
                                                # a1.txt, a2.txt ...

In [28]:
for f in audio_files:
    print(f"{f.name}: {f.stat().st_size} bytes")

meeting_hiring.mp3: 542880 bytes
meeting_product.mp3: 532320 bytes
meeting_revenue.mp3: 533760 bytes


In [29]:
parser = OpenAIWhisperParser()
sample = audio_files[0]
blob = Blob.from_path(str(sample))

In [30]:
docs = list(parser.lazy_parse(blob))

Transcribing part 1!


In [31]:
len(docs)

1

In [33]:
docs[0].page_content

'인사팀에서 채용 계획을 공유합니다. 올해 상반기에 AI 엔지니어 2명, 백엔드 개발자 15명, 데이터 분석가 10명을 채용할 예정입니다. 특히 AI 엔지니어는 멀티모달 ARG 경험자를 우대합니다. 신규 입사자 옴보딩은 4주 과정으로 진행되며 한국어와 영어 두 언어로 제공됩니다. 지원자는 회사 홈페이지의 채용 페이지를 통해 지원해 주시기 바랍니다.'

In [34]:
def transcribe_all(audio_paths):
    parser = OpenAIWhisperParser()
    out = []
    for p in audio_paths:
        blob = Blob.from_path(str(p))
        docs = list(parser.lazy_parse(blob))
        text = '\n'.join(d.page_content for d in docs)
        out.append({'source' : Path(p).name, 'text' : text})
    return out

In [35]:
results = transcribe_all(audio_files)
print(results)

Transcribing part 1!
Transcribing part 1!
Transcribing part 1!
[{'source': 'meeting_hiring.mp3', 'text': '인사팀에서 채용 계획을 공유합니다. 올해 상반기에 AI 엔지니어 2명, 백엔드 개발자 15명, 데이터 분석가 10명을 채용할 예정입니다. 특히 AI 엔지니어는 멀티모달 ARG 경험자를 우대합니다. 신규 입사자 옴보딩은 4주 과정으로 진행되며 한국어와 영어 두 언어로 제공됩니다. 지원자는 회사 홈페이지의 채용 페이지를 통해 지원해 주시기 바랍니다.'}, {'source': 'meeting_product.mp3', 'text': '제품 출시 일정을 공유 드립니다. MODUFONE X는 3월 15일 정식 출시되며 가격은 129만원으로 책정되었습니다. MODUBOOK AIR는 14인치 OLED 디스플레이를 탑재한 초경량 노트북으로 4월 1일 출시 예정입니다. MODUPAD PRO는 M.SIME 칩셋을 탑재한 프로용 태블릿으로 가격은 149만원입니다. 마케팅 캠페인은 2월 말부터 시작됩니다.'}, {'source': 'meeting_revenue.mp3', 'text': '오늘 분기 실적 회의를 시작하겠습니다. 2025년 4분기 매출은 188억 원으로 전 분기 대비 16% 증가했습니다. 특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며 가장 큰 성장을 보였습니다. 영업이익은 24억 원으로 전년 동기 대비 35% 늘었습니다. 다음 분기는 동남아시아 시장 진출 본격화할 계획입니다.'}]


In [36]:
# 어느 파일에서 왔고, 원래 뭐 (audio? )
def to_documents(transcripts):
    docs = []
    for r in transcripts:
        path = AUDIO_DIR / r['source']
        docs.append(Document(
            page_content = r['text'],
            metadata = {'source' : r['source'], 'modality' : 'audio'}
        ))
    return docs

In [37]:
audio_docs = to_documents(results)

In [38]:
audio_docs

[Document(metadata={'source': 'meeting_hiring.mp3', 'modality': 'audio'}, page_content='인사팀에서 채용 계획을 공유합니다. 올해 상반기에 AI 엔지니어 2명, 백엔드 개발자 15명, 데이터 분석가 10명을 채용할 예정입니다. 특히 AI 엔지니어는 멀티모달 ARG 경험자를 우대합니다. 신규 입사자 옴보딩은 4주 과정으로 진행되며 한국어와 영어 두 언어로 제공됩니다. 지원자는 회사 홈페이지의 채용 페이지를 통해 지원해 주시기 바랍니다.'),
 Document(metadata={'source': 'meeting_product.mp3', 'modality': 'audio'}, page_content='제품 출시 일정을 공유 드립니다. MODUFONE X는 3월 15일 정식 출시되며 가격은 129만원으로 책정되었습니다. MODUBOOK AIR는 14인치 OLED 디스플레이를 탑재한 초경량 노트북으로 4월 1일 출시 예정입니다. MODUPAD PRO는 M.SIME 칩셋을 탑재한 프로용 태블릿으로 가격은 149만원입니다. 마케팅 캠페인은 2월 말부터 시작됩니다.'),
 Document(metadata={'source': 'meeting_revenue.mp3', 'modality': 'audio'}, page_content='오늘 분기 실적 회의를 시작하겠습니다. 2025년 4분기 매출은 188억 원으로 전 분기 대비 16% 증가했습니다. 특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며 가장 큰 성장을 보였습니다. 영업이익은 24억 원으로 전년 동기 대비 35% 늘었습니다. 다음 분기는 동남아시아 시장 진출 본격화할 계획입니다.')]

In [39]:
from openai import OpenAI
oai_audio = OpenAI()

In [40]:
def transcribe_with_segments(audio_path) -> dict:
    with open(audio_path, 'rb') as f:
        res = oai_audio.audio.transcriptions.create(
                model = 'whisper-1',
                file = f,
                response_format = 'verbose_json',
                language= 'ko'
        )
        
        segs = [{'id' : s.id, 'start' : s.start, 'end' : s.end, 'text' : s.text}
                 for s in (res.segments or [])]
    
    
    return {'text' : res.text, 'segments' : segs}

In [41]:
audio2 = transcribe_with_segments(str(audio_files[0]))

In [44]:
audio2.keys()


dict_keys(['text', 'segments'])

In [45]:
def segments_to_documents(audio_path) -> list:
    tmp = transcribe_with_segments(audio_path)
    source = Path(audio_path).name
    docs = []
    for s in tmp['segments']:
        if not s['text'].strip():
            continue
        docs.append(Document(
            page_content = s['text'].strip(),
            metadata = {'source' : source, 'modality' : 'audio', 'start' : round(s['start'], 2), 'end' : round(s['end'], 2),
                       'segment_id' : s['id']}
        ))
    return docs

In [46]:
seg_docs = []
for f in audio_files:
    seg_docs.extend(segments_to_documents(str(f)))

In [47]:
len(seg_docs)

21

In [49]:
# seg_docs

In [50]:
seg_vs = FAISS.from_documents(seg_docs, embeddings)
hits = seg_vs.similarity_search('매출', k=3)
print(hits)

[Document(id='3d92bb66-27a1-4877-a14f-8bd8ba93a6ad', metadata={'source': 'meeting_revenue.mp3', 'modality': 'audio', 'start': 3.08, 'end': 6.72, 'segment_id': 1}, page_content='2025년 4분기 매출은 188억 원으로'), Document(id='05b7d935-4d1e-4d8f-8d11-792d610646ed', metadata={'source': 'meeting_revenue.mp3', 'modality': 'audio', 'start': 17.08, 'end': 19.04, 'segment_id': 5}, page_content='영업이익은 24억 원으로'), Document(id='c4da3b44-75b9-4118-8db3-8222d809422e', metadata={'source': 'meeting_revenue.mp3', 'modality': 'audio', 'start': 9.92, 'end': 14.16, 'segment_id': 3}, page_content='특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며')]


In [51]:
from pydantic import BaseModel, Field
from typing import Literal

class AudioCategory(BaseModel):
#     주제, confidence, 이유
    category : Literal['revenue', 'product', 'hiring', 'other'] = Field(..., description= '주제 카테고리')
    confidence : float = Field(..., ge=0, le=1, description="확신도 0~1")
    reason : str = Field(..., description = '한국어 짧은 사유')

In [52]:
classifier_llm = llm.with_structured_output(AudioCategory)

def classify_audio_doc(doc: Document) -> AudioCategory:
    msgs = [
        SystemMessage(content = '회의 발언을 revenue, product, hiring, other 중 하나로 분류하세요'),
        HumanMessage(content = doc.page_content)
    ]
    return classifier_llm.invoke(msgs)
    

In [53]:
for d in audio_docs:
    cat = classify_audio_doc(d)
    print(f" {d.metadata['source']} -> {cat.category} ({cat.confidence}) - {cat.reason}")

/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=AudioCategory(category='h...지고 있습니다.'), input_type=AudioCategory])
  return self.__pydantic_serializer__.to_python(


 meeting_hiring.mp3 -> hiring (0.95) - 채용 계획 및 인사 관련 내용이 중심으로 다루어지고 있습니다.


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=AudioCategory(category='p...product'로 분류됨."), input_type=AudioCategory])
  return self.__pydantic_serializer__.to_python(


 meeting_product.mp3 -> product (0.9) - 발언의 주요 내용은 제품 출시 일정과 가격에 대한 정보로, 제품 관련 내용이므로 'product'로 분류됨.
 meeting_revenue.mp3 -> revenue (0.95) - 매출 실적 및 성과에 대한 언급으로 매출 관련 내용을 다루고 있음.


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=AudioCategory(category='r...을 다루고 있음.'), input_type=AudioCategory])
  return self.__pydantic_serializer__.to_python(
